# CLINIX.AI - Brain Tumor MRI Training Pipeline

This notebook trains a custom Convolutional Neural Network (EfficientNetV2) on the Kaggle `masoudnickparvar/brain-tumor-mri-dataset` to classify brain MRI scans into 4 categories:
1. Glioma
2. Meningioma
3. Pituitary
4. No Tumor

**Instructions:**
1. Open this notebook in **Google Colab**.
2. Go to **Runtime > Change runtime type** and select **T4 GPU**.
3. Have your `kaggle.json` API file ready. You will be prompted to upload it in the first cell.
4. Run all cells (`Ctrl+F9`).
5. The final cell will export and download the trained model as `clinix_mri_model.onnx`.
6. Provide this `.onnx` file to the agent to integrate into the CLINIX.AI Java backend.

In [ ]:
!pip install kaggle torch torchvision onnx >> /dev/null

import os
from google.colab import files

print("Please upload your kaggle.json file:")
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials configured!")
    
    print("\nDownloading dataset...")
    !kaggle datasets download masoudnickparvar/brain-tumor-mri-dataset --unzip -q
    print("Dataset downloaded and extracted.")
else:
    print("Error: kaggle.json not uploaded.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time

# 1. Setup Data Paths and Parameters
TRAIN_DIR = 'Training'
TEST_DIR = 'Testing'
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
NUM_CLASSES = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# 2. Data Augmentation and Normalization
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Load Datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print(f"Classes: {class_names}")

In [ ]:
# 4. Define the Model (EfficientNet_V2_S)
weights = models.EfficientNet_V2_S_Weights.DEFAULT
model = models.efficientnet_v2_s(weights=weights)

# Replace the final classifier layer for 4 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)

# 5. Setup Optimizer and Loss Function
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.1)

# 6. Training Loop
best_acc = 0.0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print('-' * 10)
    
    for phase in ['Train', 'Val']:
        if phase == 'Train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = test_loader
            
        running_loss = 0.0
        running_corrects = 0
        
        for inputs, labels in dataloader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            
            with torch.set_grad_enabled(phase == 'Train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                
                if phase == 'Train':
                    loss.backward()
                    optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        if phase == 'Train':
            scheduler.step()
            
        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)
        
        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
        
        if phase == 'Val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            torch.save(model.state_dict(), 'best_model.pth')

print(f'\nTraining Complete. Best Validation Accuracy: {best_acc:.4f}')

In [ ]:
# 7. Export to ONNX for Java Spring Boot
print("Exporting model to ONNX format...")

# Load the best weights
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Create a dummy input tensor matching the input size
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
onnx_filename = 'clinix_mri_model.onnx'

torch.onnx.export(
    model, 
    dummy_input, 
    onnx_filename, 
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'], 
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print(f"Model exported successfully to {onnx_filename}!")

# Download the ONNX file to local machine
from google.colab import files
files.download(onnx_filename)